# 04 — Physical validation: challenge the frozen model

The model is now frozen:

\[
(T,P)\longrightarrow \widehat g_{\mathrm{OO}}(r).
\]

From this point onward we do **not** retrain, retune or redesign it. New thermodynamic states become **physical probes** of the learned structural map.

This notebook is deliberately permissive: there is no model ranking and no score to optimise. The task is to interrogate the prediction as a physicist.

In [ ]:
#@title 0. Workshop setup — run once { display-mode: "form" }
# Infrastructure is intentionally hidden so workshop time stays focused on physics.

from pathlib import Path
import hashlib, importlib.util, os, shutil, subprocess, sys, time, urllib.request, zipfile

ASSET_URL = "https://github.com/Soft-Condensed-Matter/ThermoRDF-LowData-Workshop/releases/download/student-colab-v1.0-rc1/ThermoRDF-Colab-Assets.zip"
EXPECTED_ASSET_SHA256 = "2e75fd65ad39a9dec41f7b089c2aafe51a6bb14eeee049b055be8ea3953a7bea"
EXPECTED_ASSET_VERSION = "ThermoRDF Colab Assets v1.0"
WORKSHOP_ROOT = Path("/content/ThermoRDF-Workshop")
ASSET_NAME = "ThermoRDF-Colab-Assets.zip"

def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def _download(url, destination, attempts=3):
    tmp = destination.with_suffix(destination.suffix + ".part")
    if tmp.exists():
        tmp.unlink()
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            request = urllib.request.Request(
                url,
                headers={"User-Agent": "ThermoRDF-LowData-Workshop/1.0"}
            )
            with urllib.request.urlopen(request, timeout=90) as response, open(tmp, "wb") as fh:
                shutil.copyfileobj(response, fh)
            tmp.replace(destination)
            return
        except Exception as exc:
            last_error = exc
            if tmp.exists():
                tmp.unlink()
            if attempt < attempts:
                time.sleep(2 * attempt)
    raise RuntimeError(
        "Could not download the workshop assets from GitHub. "
        "Check the internet connection and rerun this cell."
    ) from last_error

def _assets_ready(root):
    required = [
        root / "COLAB_ASSET_VERSION.txt",
        root / "data/metadata/stage_02_radial_grid.csv",
        root / "data/models/stage_07_selected_b48.pt",
        root / "data/teaching/stage_02_b48_train_40.csv.gz",
        root / "data/teaching/stage_02_b48_validation_8.csv.gz",
        root / "data/teaching/stage_02_oo_reference_372.csv.gz",
        root / "src/thermordf_workshop/__init__.py",
    ]
    if not all(path.is_file() for path in required):
        return False
    return (root / "COLAB_ASSET_VERSION.txt").read_text().strip() == EXPECTED_ASSET_VERSION

# Local/instructor override used only for automated validation.
_local_root = os.environ.get("THERMORDF_WORKSHOP_ROOT", "").strip()
if _local_root:
    WORKSHOP_ROOT = Path(_local_root).resolve()
else:
    if not _assets_ready(WORKSHOP_ROOT):
        archive = Path("/content") / ASSET_NAME

        # Reuse a verified archive if this runtime already downloaded it.
        if archive.is_file() and _sha256(archive) != EXPECTED_ASSET_SHA256:
            archive.unlink()

        if not archive.is_file():
            print("Downloading workshop assets ...")
            _download(ASSET_URL, archive)

        digest = _sha256(archive)
        if digest != EXPECTED_ASSET_SHA256:
            archive.unlink(missing_ok=True)
            raise RuntimeError(
                "Workshop asset checksum mismatch. "
                "Please rerun this cell to download a clean copy."
            )

        if WORKSHOP_ROOT.exists():
            shutil.rmtree(WORKSHOP_ROOT)
        WORKSHOP_ROOT.mkdir(parents=True)

        with zipfile.ZipFile(archive) as zf:
            zf.extractall(WORKSHOP_ROOT)

        if not _assets_ready(WORKSHOP_ROOT):
            raise RuntimeError(
                "Workshop assets were downloaded but the extracted bundle is incomplete."
            )

        print("✓ Workshop assets downloaded and verified")

_required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "scikit-learn": "sklearn",
    "torch": "torch",
}
_missing = [pkg for pkg, module in _required.items() if importlib.util.find_spec(module) is None]
if _missing:
    print("Installing missing Colab packages:", ", ".join(_missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

src_path = str(WORKSHOP_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from thermordf_workshop import *
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

torch.set_num_threads(min(2, os.cpu_count() or 1))
data = load_workshop_data(WORKSHOP_ROOT)

print(
    f"✓ Workshop ready | {len(data.train)} training + "
    f"{len(data.validation)} validation states | "
    f"{len(data.r_nm)} RDF coordinates"
)

probes = probe_states(data)
frozen_mlp = load_frozen_mlp(data)
print(f"Physical-probe pool ready | {len(probes)} non-development states")

## Choose a thermodynamic state

There are 324 simulated states that were not used in model development. Choose one that looks physically interesting: perhaps low temperature, high pressure, or a region between development points.

In [ ]:
probes.iloc[[0, 18, 80, 160, 240, 323]].reset_index(drop=True)

## 1. Predict first — before revealing the molecular reference

Edit the two values below. If you want to reveal the molecular reference later, choose an exact \((T,P)\) pair listed in `probes`.

In [ ]:
T_probe = 205
P_probe = 10

plot_probe(data, frozen_mlp, T_probe, P_probe, reveal_reference=False);

### Ask as a physicist

- Is the first shell physically plausible?
- Are the main peak and minimum well defined?
- Does the outer structure relax towards \(g(r)\approx1\)?
- Is there anything in the prediction that would make you distrust this state?

Do not try to confirm the model. Try to **stress it, question it and find its limits**.

## 2. Reveal the molecular reference

Only after forming an expectation from the model, reveal the molecular RDF for that state. Reference data are **blue symbols**; the frozen-model prediction is an **orange line**.

In [ ]:
plot_probe(data, frozen_mlp, T_probe, P_probe, reveal_reference=True);

### Interpret visually

Where does the model follow the molecular structure closely? Where does it depart? Is the disagreement localised to a structural region, or does the whole curve change character?

The goal is not to reduce this observation immediately to a metric. The goal is to decide what deserves physical attention.

## 3. Interrogate a thermodynamic path

A single state is only one probe. Follow the predicted RDF along a controlled path and look for smooth, abrupt or unexpected structural changes.

In [ ]:
plot_temperature_scan(
    data, frozen_mlp,
    P=1,
    temperatures=[205, 225, 250, 275, 300, 325, 345],
);

### Optional — change direction through thermodynamic space

Choose a temperature and explore pressure instead. You can edit both the temperature and the pressure list.

In [ ]:
plot_pressure_scan(
    data, frozen_mlp,
    T=250,
    pressures=[1, 100, 500, 1000, 1500, 2000, 3000],
);

## What is the method giving us?

The frozen network is a **fast structural surrogate**. It lets us explore thermodynamic space, formulate expectations and identify states that deserve molecular investigation.

A model prediction is not new molecular evidence. When a region looks interesting or suspicious, molecular simulation remains the physical authority.

\[
\boxed{\text{Machine learning helps decide where to look next.}}
\]